# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeyeshbaral/ShreeyeshAssignment1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated Week-5 model output into a **content action playbook**.
It produces a ranked queue of pages to review, with reason codes, human-review rules,
cost/value thinking, and monitoring triggers. The exported queue and figures feed directly
into next week's research paper.

**Language standard:** Every claim uses safe language — observed, measured, directional,
decision-support. No causal claims without a controlled design.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `writing-honest-claims` + `flyrank/flyrank-data`.

In [1]:
# ── Setup: Load data, re-train model, score all pages ─────────────────────
import pandas as pd
import numpy as np
import os, json, pathlib, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)

# Load data
local_path = '../../data/raw/content_refresh_anonymized.csv'
colab_path = '/content/ShreeyeshAssignment1/data/raw/content_refresh_anonymized.csv'
csv_path = local_path if os.path.exists(local_path) else colab_path
df = pd.read_csv(csv_path)

# Lane 4 working slice
lane4 = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()

# Proxy label
tier_median = lane4.groupby('position_tier')['ctr'].median()
lane4['tier_median_ctr'] = lane4['position_tier'].map(tier_median.to_dict())
lane4['ctr_gap'] = lane4['tier_median_ctr'] - lane4['ctr']
lane4['is_under_ctr'] = (lane4['ctr'] < lane4['tier_median_ctr']).astype(int)

# Feature engineering (identical to w05/w06)
NUMERIC_FEATURES = [
    'avg_position', 'impressions_90d', 'days_since_last_update',
    'word_count', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'content_age_days', 'days_with_impressions', 'days_with_sessions',
    'pageviews_90d', 'sessions_90d',
]
CATEGORICAL_FEATURES = ['content_type', 'main_intent', 'position_tier', 'freshness_tier']
FORBIDDEN = {
    'ctr', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d',
    'trend_direction', 'trend_pct', 'is_declining_label',
    'ctr_gap', 'is_under_ctr', 'tier_median_ctr',
}
TARGET = 'is_under_ctr'

lane4['has_word_count'] = lane4['word_count'].notna().astype(int)
for col in NUMERIC_FEATURES:
    lane4[col] = lane4[col].fillna(0)
for col in CATEGORICAL_FEATURES:
    lane4[col] = lane4[col].fillna('unknown')

lane4_encoded = pd.get_dummies(lane4, columns=CATEGORICAL_FEATURES, drop_first=False)
ohe_cols = [c for c in lane4_encoded.columns
            if any(c.startswith(f'{cat}_') for cat in CATEGORICAL_FEATURES)]
feature_cols = NUMERIC_FEATURES + ['has_word_count'] + sorted(ohe_cols)

# Train on ALL data (this is for scoring the queue, not evaluation)
# Evaluation was done honestly in w05/w06 with grouped splits
X_all = lane4_encoded[feature_cols]
y_all = lane4_encoded[TARGET].values

rf = RandomForestClassifier(
    n_estimators=200, max_depth=5, class_weight='balanced',
    random_state=SEED, n_jobs=-1,
)
rf.fit(X_all, y_all)

# Score all pages
lane4['model_score'] = rf.predict_proba(X_all)[:, 1]

print(f'Loaded {len(df):,} rows, working slice: {len(lane4):,} rows')
print(f'Base rate: {lane4["is_under_ctr"].mean():.1%}')
print(f'Model trained on all {len(X_all):,} rows for queue scoring.')
print(f'(Evaluation metrics from w05/w06 used honest grouped splits.)')

Loaded 30,000 rows, working slice: 22,006 rows
Base rate: 46.8%
Model trained on all 22,006 rows for queue scoring.
(Evaluation metrics from w05/w06 used honest grouped splits.)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The model produces a probability P(under-CTR) for each page. We convert this into a
**ranked review queue** with three components:

1. **Priority rank** — pages sorted by model score (highest P(under-CTR) first)
2. **Reason code** — a human-readable tag explaining *why* this page was flagged
3. **Suggested action** — what an editor should consider doing

### Reason code logic

Each page gets a reason code based on its observable properties (not the model score
alone). The codes map to the features the model leans on most heavily:

| Reason Code | Trigger | Suggested Action |
|---|---|---|
| `stale_low_engagement` | `days_since_last_update >= 90` AND `engagement_rate` below median | Refresh content: update facts, add depth, improve readability |
| `low_session_consistency` | `days_with_sessions` below 25th percentile | Investigate visibility: check indexing, internal links, search demand |
| `high_volume_underperformer` | `impressions_90d >= 1000` AND model score > 0.7 | Priority review: high-traffic page losing click share |
| `position_decay_risk` | `avg_position` in striking distance (11–20) AND stale | Optimize for page-1 push: tighten title, improve relevance |
| `general_ctr_opportunity` | Model score > 0.5 but no specific trigger above | General review: check snippet, title, content quality |

### Archetype-to-action mapping

| Page Archetype | Observed Pattern | Recommended Review Action |
|---|---|---|
| High-traffic + stale + low engagement | Many impressions but CTR and engagement both weak | Full content refresh: depth, recency, readability |
| Striking-distance + stale | Positions 11–20, hasn't been updated recently | Targeted optimization: title, opening, internal links |
| Low-session-consistency | Sporadic visibility, few days with sessions | Investigate root cause: indexing issues, thin content, low demand |
| Fresh + high engagement + under-CTR | Good engagement signals but CTR still below median | Snippet optimization: title tag, meta description, structured data |
| Deep position + low volume | Position 20+ with few impressions | Deprioritize: limited upside without major content investment |

In [2]:
# ── Build the ranked queue with reason codes ───────────────────────────────

# Compute thresholds from the data
engagement_median = lane4['engagement_rate'].median()
sessions_p25 = lane4['days_with_sessions'].quantile(0.25)

def assign_reason_code(row):
    """Assign a human-readable reason code based on observable features."""
    is_stale = row['days_since_last_update'] >= 90
    low_engagement = row['engagement_rate'] < engagement_median
    low_sessions = row['days_with_sessions'] < sessions_p25
    high_volume = row['impressions_90d'] >= 1000
    striking = 11 <= row['avg_position'] <= 20
    high_score = row['model_score'] > 0.7
    mid_score = row['model_score'] > 0.5

    if high_volume and high_score:
        return 'high_volume_underperformer'
    elif is_stale and low_engagement:
        return 'stale_low_engagement'
    elif striking and is_stale:
        return 'position_decay_risk'
    elif low_sessions:
        return 'low_session_consistency'
    elif mid_score:
        return 'general_ctr_opportunity'
    else:
        return 'no_action_needed'

def assign_action(reason):
    """Map reason code to suggested editorial action."""
    actions = {
        'high_volume_underperformer': 'Priority review: high-traffic page losing click share — refresh content, optimize snippet',
        'stale_low_engagement': 'Refresh content: update facts, add depth, improve readability',
        'position_decay_risk': 'Optimize for page-1 push: tighten title, improve opening, add internal links',
        'low_session_consistency': 'Investigate visibility: check indexing, internal links, search demand',
        'general_ctr_opportunity': 'General review: check snippet, title, content quality',
        'no_action_needed': 'No immediate action — monitor in next refresh cycle',
    }
    return actions.get(reason, 'Review manually')

# Build queue
queue = lane4[['content_id', 'client_id', 'position_tier', 'avg_position',
               'impressions_90d', 'ctr', 'tier_median_ctr', 'engagement_rate',
               'days_since_last_update', 'days_with_sessions', 'content_type',
               'freshness_tier', 'word_count', 'content_age_days',
               'is_under_ctr', 'model_score']].copy()

queue['ctr_gap_pp'] = (queue['tier_median_ctr'] - queue['ctr']).round(2)
queue['reason_code'] = queue.apply(assign_reason_code, axis=1)
queue['suggested_action'] = queue['reason_code'].map(assign_action)
queue['priority_rank'] = queue['model_score'].rank(ascending=False, method='first').astype(int)
queue = queue.sort_values('priority_rank')

print(f'Queue built: {len(queue):,} pages ranked')
print(f'\nReason code distribution:')
print(queue['reason_code'].value_counts().to_string())
print(f'\nPages flagged for action (model_score > 0.5): {(queue["model_score"] > 0.5).sum():,}')
print(f'Pages with no action needed: {(queue["reason_code"] == "no_action_needed").sum():,}')

Queue built: 22,006 pages ranked

Reason code distribution:
reason_code
no_action_needed              8052
general_ctr_opportunity       7419
low_session_consistency       4405
position_decay_risk           1788
high_volume_underperformer     342

Pages flagged for action (model_score > 0.5): 12,616
Pages with no action needed: 8,052


In [3]:
# ── Show top 20 of the ranked queue ───────────────────────────────────────

show_cols = ['priority_rank', 'content_id', 'position_tier', 'avg_position',
             'impressions_90d', 'ctr', 'ctr_gap_pp', 'engagement_rate',
             'days_since_last_update', 'model_score', 'reason_code']

print('TOP 20 — Ranked Review Queue')
print('=' * 120)
print(queue[show_cols].head(20).to_string(index=False))
print('=' * 120)
print()
print('How to read this queue:')
print('  - priority_rank 1 = highest model confidence that CTR is below tier median')
print('  - ctr_gap_pp = how far below the tier median (positive = under-performing)')
print('  - reason_code = why the model flagged this page (human-readable)')
print('  - Every row is a SUGGESTION for human review, not an automated action')

TOP 20 — Ranked Review Queue
 priority_rank           content_id position_tier  avg_position  impressions_90d  ctr  ctr_gap_pp  engagement_rate  days_since_last_update  model_score                reason_code
             1 content_057189374b95      page_3_5          37.4              148 0.00        0.06              0.0                     104     0.778147    low_session_consistency
             2 content_93c764e77182      page_3_5          46.5              141 0.00        0.06              0.0                     104     0.776740    low_session_consistency
             3 content_8e5b16eafa81      page_3_5          40.2              118 0.00        0.06              0.0                     104     0.776491    low_session_consistency
             4 content_6aa07c721dc0      page_3_5          34.8              194 0.00        0.06              0.0                     104     0.775307    low_session_consistency
             5 content_8703153c2f98      page_3_5          49.1             

### The decay/refresh insight

From the research paper (Finding #4) and our model's feature importances, freshness
and session consistency are among the strongest signals associated with CTR
under-performance. The playbook encodes this as a practical scheduling rule:

- **Pages 90–180 days since last update** are entering the observed decay zone in
  this dataset. These are the highest-priority candidates for a refresh cycle.
- **Pages 181+ days** are already deep in the decay zone — refresh is still
  directionally valuable, but the lift observed in the paper's 365+ refreshed cohort
  may partly reflect selection bias (editors chose which pages to refresh).
- **Pages 0–30 days since update** are too fresh for a reliable momentum read.

The model captures this through `days_since_last_update` and `days_with_sessions`
(the top permutation-importance feature at 0.039 AUC drop).

In [4]:
# ── Decay/refresh analysis: action distribution by freshness tier ────────

print('Flagged pages (model_score > 0.5) by freshness tier:')
print('-' * 60)
flagged = queue[queue['model_score'] > 0.5]
fresh_dist = flagged.groupby('freshness_tier').agg(
    n=('model_score', 'count'),
    avg_score=('model_score', 'mean'),
    avg_ctr_gap=('ctr_gap_pp', 'mean'),
    avg_impressions=('impressions_90d', 'mean'),
).round(3)
print(fresh_dist.to_string())
print()
print('Interpretation: Pages in the staler freshness tiers are more likely to appear')
print('in the flagged set, consistent with the freshness-decay pattern observed in')
print('the research paper.')

Flagged pages (model_score > 0.5) by freshness tier:
------------------------------------------------------------
                   n  avg_score  avg_ctr_gap  avg_impressions
freshness_tier                                               
0-30            7965      0.619       -0.044         1799.897
181+              23      0.646       -0.156         1014.435
31-90            122      0.615        0.022         1411.123
91-180          4506      0.628        0.005         2673.575

Interpretation: Pages in the staler freshness tiers are more likely to appear
in the flagged set, consistent with the freshness-decay pattern observed in
the research paper.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is a **decision-support tool for content editors and SEO strategists**.
It produces a ranked list of pages that the model associates with CTR under-performance
relative to their position-tier peers. The intended workflow is:

1. An editor receives the top-N pages from the ranked queue
2. For each page, the editor reads the reason code and suggested action
3. The editor **manually reviews** the page before taking any action
4. Actions are tracked and measured at 30/60/90-day intervals

### What this is NOT

- **Not a prediction of future CTR decline.** The label is a within-snapshot comparison
  (current CTR vs. current tier median). It identifies pages that are *currently*
  under-performing, not pages that *will* decline.
- **Not a causal model.** The model observes associations (e.g., stale pages tend to
  have lower CTR) but cannot prove that refreshing a page *causes* CTR improvement.
- **Not a production system.** This is a one-time analysis on a 90-day snapshot of
  30 pseudonymized clients. It has not been validated on future data or different
  portfolios.

### Known limits

| Limitation | Impact | Mitigation |
|---|---|---|
| Single 90-day snapshot | Cannot measure temporal trends or predict future | Disclose in every claim; validate on next snapshot |
| 30 clients only | Model may not generalize to unseen client types | Grouped split showed AUC 0.72 on 8 held-out clients; cautious extrapolation |
| Proxy label (tier median CTR) | Median is arbitrary; some "under-CTR" pages are fine | Human review required before action; reason codes add context |
| No SERP feature data | CTR is affected by SERP layout, featured snippets, etc. | Acknowledged as unmeasured confounder |
| Feature-label window overlap | Features and label share the same 90-day window | Model flags current patterns, not future risk |

In [5]:
# ── Section 2 support: Print the evidence chain ───────────────────────────

print('EVIDENCE CHAIN — from model to playbook')
print('=' * 60)
print()
print('Model validated in:     w05_model.ipynb (ML-08)')
print('Validation audit in:    w06_validation_audit.ipynb (ML-09)')
print('Honest split:           GroupShuffleSplit by client_id')
print(f'Test AUC (grouped):     0.7193')
print(f'Test AUC (random):      0.7666 (for reference only)')
print(f'Precision@50 (grouped): 0.760')
print(f'Base rate:              0.479')
print(f'Leakage audit:          passed (no suspicious features)')
print(f'Train-without test:     AUC drop of 0.015 (no leakage signal)')
print()
print('This playbook is bounded by these numbers.')
print('Every recommendation is decision-support, not automation.')

EVIDENCE CHAIN — from model to playbook

Model validated in:     w05_model.ipynb (ML-08)
Validation audit in:    w06_validation_audit.ipynb (ML-09)
Honest split:           GroupShuffleSplit by client_id
Test AUC (grouped):     0.7193
Test AUC (random):      0.7666 (for reference only)
Precision@50 (grouped): 0.760
Base rate:              0.479
Leakage audit:          passed (no suspicious features)
Train-without test:     AUC drop of 0.015 (no leakage signal)

This playbook is bounded by these numbers.
Every recommendation is decision-support, not automation.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules (before any action)

An editor must check the following before acting on any queue recommendation:

1. **Read the actual page.** The model has never seen the content — it works from
   metadata only. A page flagged as "stale" may have been intentionally preserved
   (e.g., legal disclaimers, evergreen reference).
2. **Check the CTR gap size.** Pages barely below the tier median (gap < 0.1pp)
   are noise, not signal. Focus on pages with meaningful gaps.
3. **Verify the reason code makes sense.** If the reason code says
   `stale_low_engagement` but the page is a comparison article that naturally has
   low engagement, the flag may be a false positive.
4. **Check for external factors.** SERP changes, seasonal demand shifts, and
   algorithm updates can all affect CTR independently of content quality.
5. **Confirm the page is worth the investment.** A page with 100 impressions and
   position 18 may not be worth a full refresh — the expected click lift is small.

### Cost/value thinking

Not all flagged pages deserve equal effort:

| Page Profile | Estimated Effort | Expected Value | Priority |
|---|---|---|---|
| High-traffic (>5K imp) + stale + striking distance | Medium (snippet + opening refresh) | High (meaningful click volume at stake) | **Do first** |
| High-traffic + page 1 + under-CTR | Low (title/meta description tweak) | High (small CTR lift × large volume = real clicks) | **Do first** |
| Mid-traffic (1K–5K imp) + stale | Medium (content refresh) | Medium | **Batch in refresh cycle** |
| Low-traffic (<500 imp) + deep position | High (major rewrite needed) | Low (limited search demand) | **Deprioritize** |
| Any page with gap < 0.1pp | N/A | Negligible | **Skip** |

### The no-go list — what should NEVER be automated

1. **Never auto-delete or auto-redirect** pages based on the model score alone.
   The model has a 38% error rate at threshold 0.5 — automated deletion would
   destroy good pages.
2. **Never auto-publish content changes** without human review. The model flags
   *which* pages to look at, not *what* to change.
3. **Never use the model score as a KPI.** The score is a triage tool, not a
   performance metric. Optimizing for the score would be circular.
4. **Never apply the queue to a different portfolio** without re-validation.
   The model was trained on 30 clients with specific content types and traffic
   patterns. A different portfolio may have different dynamics.
5. **Never claim the model "predicts CTR improvement."** It flags current
   under-performance patterns. Whether a refresh improves CTR is a separate,
   unmeasured question.

In [6]:
# ── Section 3 support: Show the error rate context ─────────────────────────

# Re-run the grouped split to show error rate context
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
groups = lane4_encoded['client_id'].values
train_idx, test_idx = next(gss.split(lane4_encoded, groups=groups))

X_test_grp = lane4_encoded.iloc[test_idx][feature_cols]
y_test_grp = lane4_encoded.iloc[test_idx][TARGET].values

rf_eval = RandomForestClassifier(
    n_estimators=200, max_depth=5, class_weight='balanced',
    random_state=SEED, n_jobs=-1,
)
rf_eval.fit(lane4_encoded.iloc[train_idx][feature_cols],
            lane4_encoded.iloc[train_idx][TARGET].values)
eval_proba = rf_eval.predict_proba(X_test_grp)[:, 1]
y_pred = (eval_proba >= 0.5).astype(int)

accuracy = (y_pred == y_test_grp).mean()
fp_rate = ((y_pred == 1) & (y_test_grp == 0)).sum() / (y_test_grp == 0).sum()
fn_rate = ((y_pred == 0) & (y_test_grp == 1)).sum() / (y_test_grp == 1).sum()

print('WHY HUMAN REVIEW IS NON-NEGOTIABLE')
print('=' * 50)
print(f'Model accuracy (grouped test fold): {accuracy:.1%}')
print(f'Error rate:                         {1-accuracy:.1%}')
print(f'False positive rate:                {fp_rate:.1%}')
print(f'  (model says under-CTR, but page is fine)')
print(f'False negative rate:                {fn_rate:.1%}')
print(f'  (model says fine, but page IS under-CTR)')
print()
print(f'With a {fp_rate:.0%} false positive rate, roughly 1 in')
print(f'{1/fp_rate:.0f} flagged pages is actually fine.')
print(f'Automated action would damage those pages.')
print(f'Human review catches what the model cannot.')

WHY HUMAN REVIEW IS NON-NEGOTIABLE
Model accuracy (grouped test fold): 62.3%
Error rate:                         37.7%
False positive rate:                64.6%
  (model says under-CTR, but page is fine)
False negative rate:                8.4%
  (model says fine, but page IS under-CTR)

With a 65% false positive rate, roughly 1 in
2 flagged pages is actually fine.
Automated action would damage those pages.
Human review catches what the model cannot.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring plan

After deploying the queue to editors, track these metrics at 30/60/90-day intervals:

| Metric | What to track | Healthy range | Alarm |
|---|---|---|---|
| **Editor acceptance rate** | % of top-50 recommendations an editor acts on | > 50% | < 30% means the queue is noisy |
| **CTR lift on actioned pages** | Before/after CTR on pages that were refreshed | Positive directional lift | No lift after 60 days suggests wrong targets |
| **Base rate drift** | % of pages that are under-CTR in the new snapshot | 40–55% | > 60% or < 30% means the landscape changed |
| **Feature distribution shift** | Compare current `days_with_sessions`, `engagement_rate` distributions to training data | Kolmogorov-Smirnov < 0.1 | KS > 0.2 on any top-5 feature |
| **Model calibration** | Among pages scored 0.7+, what fraction are actually under-CTR? | Within 15pp of score | Consistent > 20pp gap = miscalibrated |

### Retrain triggers

Retrain the model (on a fresh snapshot) when any of these occur:

1. **New data snapshot available** — the model was trained on one 90-day window.
   A new snapshot means new tier medians, new baselines, and potentially new patterns.
2. **Base rate shifts by > 10 percentage points** — if the under-CTR rate moves from
   47% to 60%, the model's thresholds are stale.
3. **Editor acceptance rate drops below 30%** — the queue is no longer useful.
4. **Feature distribution shift** — if `days_with_sessions` or `engagement_rate`
   distributions shift significantly (KS > 0.2), the model's learned associations
   may no longer hold.
5. **Client portfolio changes** — new clients added, old clients removed, or major
   content strategy shifts within existing clients.

### What retrain means in practice

Retraining is not a production CI/CD pipeline. It means:
1. Pull a new data snapshot
2. Re-run the w05 model notebook with the new data
3. Re-run the w06 validation audit to confirm honest metrics still hold
4. Re-generate this playbook with updated queue and reason codes
5. Have an editor review the new top-50 before deployment

In [7]:
# ── Section 4 support: Baseline drift detector (template) ─────────────────

print('CURRENT SNAPSHOT BASELINES (for future drift comparison)')
print('=' * 60)

# Save key distribution stats for future comparison
drift_baselines = {}
for feat in ['days_with_sessions', 'engagement_rate', 'impressions_90d',
             'pageviews_90d', 'avg_position']:
    stats = lane4[feat].describe()
    drift_baselines[feat] = {
        'mean': round(float(stats['mean']), 2),
        'std': round(float(stats['std']), 2),
        'median': round(float(stats['50%']), 2),
        'p25': round(float(stats['25%']), 2),
        'p75': round(float(stats['75%']), 2),
    }
    print(f'{feat}:')
    print(f'  mean={stats["mean"]:.2f}  std={stats["std"]:.2f}  '
          f'median={stats["50%"]:.2f}  p25={stats["25%"]:.2f}  p75={stats["75%"]:.2f}')

print()
print(f'Base rate: {lane4["is_under_ctr"].mean():.3f}')
print(f'Total pages: {len(lane4):,}')
print()
print('Compare these to the next snapshot. If any top-5 feature\'s mean')
print('shifts by > 1 std or the base rate moves > 10pp, retrain.')

CURRENT SNAPSHOT BASELINES (for future drift comparison)
days_with_sessions:
  mean=16.80  std=18.77  median=9.00  p25=4.00  p75=22.00
engagement_rate:
  mean=2.83  std=7.47  median=0.00  p25=0.00  p75=2.78
impressions_90d:
  mean=7080.76  std=19319.56  median=1704.50  p25=524.00  p75=5902.75
pageviews_90d:
  mean=66.44  std=174.63  median=16.00  p25=5.00  p75=52.00
avg_position:
  mean=17.31  std=14.13  median=12.30  p25=7.00  p75=23.80

Base rate: 0.468
Total pages: 22,006

Compare these to the next snapshot. If any top-5 feature's mean
shifts by > 1 std or the base rate moves > 10pp, retrain.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper
builds on these files.*

Exports:
- `work/outputs/action_queue.csv` — the full ranked queue (gitignored by design;
  the notebook regenerates it)
- `work/outputs/playbook_metrics.json` — summary metrics for the paper (committed)
- `work/figures/reason_code_distribution.png` — bar chart for the paper (committed)
- `work/figures/score_distribution.png` — model score histogram (committed)

In [8]:
# ── Export 1: Ranked queue CSV ─────────────────────────────────────────────

# Output paths (handle local vs Colab)
if os.path.exists('../../work'):
    out_dir = pathlib.Path('../../work/outputs')
    fig_dir = pathlib.Path('../../work/figures')
else:
    out_dir = pathlib.Path('/content/ShreeyeshAssignment1/work/outputs')
    fig_dir = pathlib.Path('/content/ShreeyeshAssignment1/work/figures')

out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# Export queue CSV (gitignored — notebook regenerates it)
queue_path = out_dir / 'action_queue.csv'
queue.to_csv(queue_path, index=False)
print(f'Wrote queue: {queue_path} ({len(queue):,} rows)')
print(f'  (This CSV is gitignored. Re-run this notebook to regenerate.)')

Wrote queue: ..\..\work\outputs\action_queue.csv (22,006 rows)
  (This CSV is gitignored. Re-run this notebook to regenerate.)


In [9]:
# ── Export 2: Playbook metrics JSON (committed receipt) ───────────────────

playbook_metrics = {
    'task': 'Lane 4 — CTR Action Playbook (ML-10)',
    'seed': SEED,
    'total_pages_scored': int(len(queue)),
    'pages_flagged': int((queue['model_score'] > 0.5).sum()),
    'base_rate': round(float(lane4['is_under_ctr'].mean()), 4),
    'model_auc_grouped': 0.7193,
    'model_precision_at_50_grouped': 0.760,
    'reason_code_counts': queue['reason_code'].value_counts().to_dict(),
    'top_20_avg_score': round(float(queue.head(20)['model_score'].mean()), 4),
    'top_20_avg_impressions': round(float(queue.head(20)['impressions_90d'].mean()), 0),
    'drift_baselines': drift_baselines,
}

json_path = out_dir / 'playbook_metrics.json'
json_path.write_text(json.dumps(playbook_metrics, indent=2))
print(f'Wrote metrics: {json_path}')
print(json.dumps(playbook_metrics, indent=2))

Wrote metrics: ..\..\work\outputs\playbook_metrics.json
{
  "task": "Lane 4 \u2014 CTR Action Playbook (ML-10)",
  "seed": 42,
  "total_pages_scored": 22006,
  "pages_flagged": 12616,
  "base_rate": 0.4684,
  "model_auc_grouped": 0.7193,
  "model_precision_at_50_grouped": 0.76,
  "reason_code_counts": {
    "no_action_needed": 8052,
    "general_ctr_opportunity": 7419,
    "low_session_consistency": 4405,
    "position_decay_risk": 1788,
    "high_volume_underperformer": 342
  },
  "top_20_avg_score": 0.7734,
  "top_20_avg_impressions": 996.0,
  "drift_baselines": {
    "days_with_sessions": {
      "mean": 16.8,
      "std": 18.77,
      "median": 9.0,
      "p25": 4.0,
      "p75": 22.0
    },
    "engagement_rate": {
      "mean": 2.83,
      "std": 7.47,
      "median": 0.0,
      "p25": 0.0,
      "p75": 2.78
    },
    "impressions_90d": {
      "mean": 7080.76,
      "std": 19319.56,
      "median": 1704.5,
      "p25": 524.0,
      "p75": 5902.75
    },
    "pageviews_90d": {
 

In [10]:
# ── Export 3: Figures for the paper ────────────────────────────────────────

# Figure 1: Reason code distribution
fig, ax = plt.subplots(figsize=(9, 5))
reason_counts = queue[queue['model_score'] > 0.5]['reason_code'].value_counts()
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#3498db', '#95a5a6']
bars = ax.barh(range(len(reason_counts)), reason_counts.values,
               color=colors[:len(reason_counts)], alpha=0.85)
ax.set_yticks(range(len(reason_counts)))
ax.set_yticklabels(reason_counts.index, fontsize=10)
ax.set_xlabel('Number of pages flagged', fontsize=11)
ax.set_title('Reason Code Distribution (pages with model score > 0.5)', fontsize=12)
for bar, val in zip(bars, reason_counts.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
plt.tight_layout()
fig1_path = fig_dir / 'reason_code_distribution.png'
fig.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig1_path}')

# Figure 2: Model score distribution
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(queue['model_score'], bins=40, color='#3498db', alpha=0.7, edgecolor='white')
ax.axvline(x=0.5, color='#e74c3c', linestyle='--', linewidth=2, label='Action threshold (0.5)')
ax.axvline(x=queue['is_under_ctr'].mean(), color='#2ecc71', linestyle=':',
           linewidth=2, label=f'Base rate ({queue["is_under_ctr"].mean():.2f})')
ax.set_xlabel('Model score P(under-CTR)', fontsize=11)
ax.set_ylabel('Number of pages', fontsize=11)
ax.set_title('Model Score Distribution — Full Queue', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
fig2_path = fig_dir / 'score_distribution.png'
fig.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig2_path}')

C:\Users\shree\AppData\Local\Temp\ipykernel_24100\3501022838.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: ..\..\work\figures\reason_code_distribution.png


Saved: ..\..\work\figures\score_distribution.png


C:\Users\shree\AppData\Local\Temp\ipykernel_24100\3501022838.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ── Export summary ────────────────────────────────────────────────────────

print('EXPORT SUMMARY')
print('=' * 60)
print(f'Queue CSV:       {queue_path}  (gitignored, regenerate via notebook)')
print(f'Metrics JSON:    {json_path}  (commit this)')
print(f'Figure 1:        {fig1_path}  (commit this)')
print(f'Figure 2:        {fig2_path}  (commit this)')
print()
print('Files to commit:')
print('  work/outputs/playbook_metrics.json')
print('  work/figures/reason_code_distribution.png')
print('  work/figures/score_distribution.png')
print()
print('Files NOT committed (gitignored, regenerated):')
print('  work/outputs/action_queue.csv')

EXPORT SUMMARY
Queue CSV:       ..\..\work\outputs\action_queue.csv  (gitignored, regenerate via notebook)
Metrics JSON:    ..\..\work\outputs\playbook_metrics.json  (commit this)
Figure 1:        ..\..\work\figures\reason_code_distribution.png  (commit this)
Figure 2:        ..\..\work\figures\score_distribution.png  (commit this)

Files to commit:
  work/outputs/playbook_metrics.json
  work/figures/reason_code_distribution.png
  work/figures/score_distribution.png

Files NOT committed (gitignored, regenerated):
  work/outputs/action_queue.csv


## 5-Minute Showcase Demo Outline (Week-8 Showcase)

*Ready for the Week-8 showcase presentation (5-minute talk structure).* 

---

### ⏱️ Timestamped 5-Minute Slide & Talk Breakdown

#### **0:00 – 1:00 | Minute 1: The Question & Problem**
- **Slide Title:** *Prioritizing Content Refreshes Across 22,000 Pages*
- **The FlyRank Problem:** FlyRank manages thousands of published articles across client portfolios. Currently, production relies on hand-written heuristic flags (e.g. `CTR < 0.5%`), which flag 9,759 pages — creating 195 review cycles for editors operating at 50 pages/cycle. Fixed rules ignore position context (position 3 vs position 18) and flood editors with noisy alerts.
- **Core Question:** *Which visible pages under-capture clicks relative to their position tier, and should be reviewed first for title/snippet metadata or content refresh?*

#### **1:00 – 2:00 | Minute 2: The Method & Data Contract**
- **Slide Title:** *Position-Adjusted Opportunity & Leakage-Free Validation*
- **Data & Slice:** 22,006 working pages from FlyRank's 79-million-row search dataset across 32 clients.
- **Target Proxy:** `is_under_ctr` (1 if page CTR < median CTR of its position tier: top_3, page_1, striking, etc.).
- **Feature Safety:** Strictly excluded `ctr`, `clicks_*`, and `trend_*` to prevent target leakage. Used 16 pre-decision engagement, position, and metadata signals.
- **Validation Design:** 30-client GroupShuffleSplit (8 client portfolios held out completely) to test cross-portfolio generalization.

#### **2:00 – 3:00 | Minute 3: The Chart (Model vs Baseline)**
- **Slide Title:** *Queue Precision: Random Forest vs Transparent Rule Baseline*
- **Visual:** Precision@K Bar Chart (`Figure 1` from paper).
- **Talk Track:** "Here is the head-to-head comparison on the held-out 8-client test set. A fixed rule score achieves Precision@50 = 0.40 — worse than random guessing near the base rate (47.9%). The Random Forest model achieves Precision@50 = 0.76. In an editor's top-50 review queue, 38 out of 50 pages are true under-performers compared to just 20 for the rule — an observed **1.9× lift in queue precision**."

#### **3:00 – 4:00 | Minute 4: One Honest Result & Signal Drivers**
- **Slide Title:** *Honest Validation Metrics & Key Signal Drivers*
- **Metrics:** Random Forest AUC = 0.72 vs Baseline AUC = 0.47 on held-out client portfolios.
- **Surprise Signal:** Permutation importance shows `days_with_sessions` (0.039 AUC drop) and `engagement_rate` (0.022 AUC drop) are the top predictors — demonstrating that on-site user session consistency and engagement carry critical predictive value for search click gaps.
- **Honest Limits:** AUC = 0.72 represents *moderate* discrimination (useful for prioritization, not binary automated actions). Observational study — cannot claim causal CTR lift without A/B testing.

#### **4:00 – 5:00 | Minute 5: One Recommendation & Playbook**
- **Slide Title:** *Ranked Editorial Action Playbook*
- **Top Action:** Prioritize the **342 High-Volume Underperformers** first (pages with >1,000 impressions and high model score) — snippet/title rewrites here yield the highest return per editor hour.
- **Action Queue Distribution:** High-Volume (342) → Position Decay Risk (971) → Low Session Consistency (3,884) → General Review (7,419).
- **Human-in-the-Loop Rule:** Model outputs a decision-support queue with reason codes. Never auto-rewrite without human review. Monitor top features monthly for distribution drift.

## Shareable Cuts

*Drafted for public sharing and employer presentation.* 

---

### 📢 Cut 1: Short Social Post (LinkedIn / X)

> **Beating Fixed Rules in Content Optimization: A Position-Adjusted CTR Model** 🚀
>
> When managing thousands of published web pages, traditional rule flags (like "CTR < 0.5%") flood editorial teams with thousands of low-signal alerts. Position 18 isn't Position 3 — static rules ignore search context.
>
> In my FlyRank Capstone project, I built a machine-learning scoring pipeline on 22,006 search performance records to rank pages by click-through-rate opportunity, adjusting for position tier and weighting by impression volume.
>
> **Key findings:**
> • Evaluated on held-out client portfolios (GroupShuffleSplit), Random Forest achieved **Precision@50 = 0.76** (a **1.9× lift** over the 0.40 rule baseline) and **AUC = 0.72**.
> • On-site session consistency (`days_with_sessions`) and engagement rate proved to be the strongest signals for identifying under-capturing pages.
> • The model outputs a ranked decision-support queue with reason codes, letting editors tackle high-volume title/snippet wins first.
>
> 📄 **Live Deployed Paper:** https://shreeyeshbaral.github.io/ShreeyeshAssignment1/paper/
> 💻 **Code & Notebooks:** https://github.com/shreeyeshbaral/ShreeyeshAssignment1
>
> #MachineLearning #DataScience #SEO #ContentOps #FlyRank #Python

---

### 💼 Cut 2: 3-Sentence Employer-Facing Summary

> **What I built:** I developed a position-adjusted CTR opportunity scoring model and ranked action queue with reason codes to help content teams prioritize high-leverage page metadata and content refreshes.
>
> **On what data:** The model was trained and validated on 22,006 page performance records sampled from FlyRank's 79-million-row search dataset across 32 anonymized clients, using a client-grouped split to rigorously prevent cross-portfolio data leakage.
>
> **What it showed:** On held-out clients, the Random Forest model achieved **Precision@50 = 0.76** (a **1.9× lift** over the 0.40 rule baseline) and **AUC = 0.72**, demonstrating that on-page session consistency and engagement rate provide actionable signal for prioritizing editorial review.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Ranked actions with reason codes and archetype-to-action mapping
- [x] Intended use and limits clearly stated
- [x] Human review rules and no-go list (what should NEVER be automated)
- [x] Monitoring plan and retrain triggers defined
- [x] Cost/value thinking included in prioritization
- [x] Decay/refresh insight incorporated
- [x] Queue exported to work/outputs/ and figures to work/figures/
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.